# 02b — Multi-Agent Orchestration
**Block B · SAPA AI Workshop**

In `02a` you built a single agent with one tool.
Now you will wire **three specialized agents** into a LangGraph pipeline.

Notice that the agent code is **imported from the `agents/` folder** — not written here.
This is intentional: real systems keep agents as separate, reusable modules.
Your job here is to understand how they are **composed**, not how each one works internally.

```
agents/
  tools.py            ← calculate_severity_score, lookup_drug_class, check_required_fields
  severity_agent.py   ← severity_node(state) -> state update
  compliance_agent.py ← compliance_node(state) -> state update
  summary_agent.py    ← summary_node(state) -> state update
```

```mermaid
graph LR
    Input --> severity_node --> compliance_node --> summary_node --> Output
```

---
## Step 1 — Setup

In [ ]:
import os, sys
sys.path.insert(0, '..')  # make the agents/ folder importable from notebooks/

from dotenv import load_dotenv
load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set"
print("Ready.")

---
## Step 2 — Define the Shared State

State is the **shared clipboard** that flows through every node in the graph.
Every agent reads from it and writes its findings back to it.

Think of it as the running case file that gets handed from specialist to specialist.

In [ ]:
from typing import TypedDict

class AEReportState(TypedDict):
    report:              str   # input — the raw adverse event report
    severity_analysis:   str   # written by severity_node
    compliance_analysis: str   # written by compliance_node
    final_report:        str   # written by summary_node

print("State schema defined:", list(AEReportState.__annotations__.keys()))

---
## Step 3 — Import the Agent Nodes

Each agent is a plain Python function: `node(state) -> dict`
- It receives the current State
- It returns a dict with only the keys it updated

LangGraph merges the returned dict back into the State automatically.

In [ ]:
from agents.severity_agent import severity_node
from agents.compliance_agent import compliance_node
from agents.summary_agent import summary_node

print("Agents imported:")
print("  severity_node   — scores severity, looks up drug class")
print("  compliance_node — checks required ICH E2B fields")
print("  summary_node    — synthesizes findings into a structured report")

---
## Step 4 — Build the Graph

This is the orchestration layer. Three lines of code per agent:
1. `add_node` — register the node
2. `add_edge` — define the handoff
3. `compile` — freeze the graph into a callable

To add a new specialist: add one node + one edge. Nothing else changes.

In [ ]:
from langgraph.graph import StateGraph, END

# 1. Initialize the graph with our State schema
builder = StateGraph(AEReportState)

# 2. Register nodes
builder.add_node("severity",   severity_node)
builder.add_node("compliance", compliance_node)
builder.add_node("summary",    summary_node)

# 3. Define edges (fixed sequential flow)
builder.set_entry_point("severity")
builder.add_edge("severity",   "compliance")
builder.add_edge("compliance", "summary")
builder.add_edge("summary",    END)

# 4. Compile
pipeline = builder.compile()
print("Pipeline compiled. Nodes:", list(builder.nodes.keys()))

---
## Step 5 — Run the Pipeline

Feed in a synthetic adverse event report and watch the three agents process it in sequence.

In [ ]:
SAMPLE_REPORT = """
Patient: 45-year-old female.
Drug: Metformin 500mg twice daily.
Onset: 2024-01-15.
Adverse event: Severe nausea, vomiting, and lactic acidosis.
Outcome: Hospitalized for 3 days, recovered.
Reporter: Prescribing physician.
"""

print("Running pipeline...\n")

# Invoke the graph — returns the final State
final_state = pipeline.invoke({
    "report": SAMPLE_REPORT,
    "severity_analysis": "",
    "compliance_analysis": "",
    "final_report": "",
})

print("Pipeline complete.")

---
## Step 6 — Inspect the Results

Because everything flows through shared State, you can inspect what each agent produced independently.

In [ ]:
print("=" * 60)
print("SEVERITY AGENT OUTPUT")
print("=" * 60)
print(final_state["severity_analysis"])

print()
print("=" * 60)
print("COMPLIANCE AGENT OUTPUT")
print("=" * 60)
print(final_state["compliance_analysis"])

print()
print("=" * 60)
print("FINAL STRUCTURED REPORT (Summary Agent)")
print("=" * 60)
print(final_state["final_report"])

---
## What You Just Built

| Component | What it is | Where it lives |
|-----------|-----------|----------------|
| `AEReportState` | Shared clipboard | Defined above |
| `severity_node` | Agent + 2 tools | `agents/severity_agent.py` |
| `compliance_node` | Agent + 1 tool | `agents/compliance_agent.py` |
| `summary_node` | Agent, no tools | `agents/summary_agent.py` |
| `pipeline` | The compiled graph | Assembled above |

**To add a new agent in Block D:**
1. Create `agents/your_agent.py` with a `your_node(state) -> dict` function
2. `builder.add_node("your_agent", your_node)`
3. `builder.add_edge("compliance", "your_agent")` and `builder.add_edge("your_agent", "summary")`
4. Recompile

That's it. The rest of the pipeline is unchanged.

> **Up next:** Block C — Vibe Coding & Agentic Development Patterns